# Gaussian LiRA v2 -- Proper Likelihood Ratio Attack

**EECE 608 -- Mohamad Faour**

This notebook implements the **proper Gaussian LiRA** from Carlini et al. (2022):
- Train K shadow models on random 50% subsets of training data
- For each target example, fit Gaussians to IN-model losses and OUT-model losses
- Score = log(pdf_out(loss) / pdf_in(loss)) -- the actual likelihood ratio
- Sweep K = 8, 16, 32, 64 to show improvement with more shadow models

**Key difference from v1:** The previous notebook used raw `mean(OUT) - mean(IN)` as the score.
This notebook fits per-example Gaussians and computes the proper log-likelihood ratio,
which is much more statistically powerful.

### Setup
1. Runtime -> Change runtime type -> **L4 GPU** (or T4)
2. Click **Run All** (Runtime -> Run all)
3. Total runtime: ~15 min for K=64

---
## 0. Clone repo and install

In [ ]:
import os, sys, shutil
from pathlib import Path

# Clone repo onto Colab's machine (GPU server, not your laptop)
WORK = Path("/content/EECE_608")
if not WORK.exists():
    !git clone https://github.com/mohdfaour03/EECE_608.git /content/EECE_608

os.chdir(WORK)
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml
!pip install -e . -q

sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK / "autoresearch"))

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Ready.")

## 1. Load target model and imports

In [ ]:
import prepare
import torch
import torch.nn.functional as F
import random
import time
import json
import statistics
import logging
import numpy as np
from pathlib import Path
from scipy import stats as scipy_stats
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.data.datasets import DatasetBundle
from dp_audit_tightness.utils.seeds import set_global_seed
from dp_audit_tightness.models.io import load_model_for_inference
from torch.utils.data import Subset

logging.basicConfig(level=logging.INFO, format='%(message)s')

# Train/load target model
model, bundle, device, record = prepare.load_model_and_data()
EPS_UPPER = record.epsilon_upper_theory
print(f"Target model: epsilon_upper={EPS_UPPER:.4f}")
print(f"Device: {device}")
print(f"Train set: {len(bundle.train_dataset)}, Eval set: {len(bundle.eval_dataset)}")

In [ ]:
# --- Helper functions ---

def collect_logits(mdl, dataset, indices, dev, batch_size=512):
    """Get model logits and labels for given indices."""
    logits_list, labels_list = [], []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            bi = indices[s:s+batch_size]
            imgs, lbls = zip(*(dataset[i] for i in bi))
            x = torch.stack(imgs).to(dev)
            y = torch.tensor(lbls, dtype=torch.long, device=dev)
            logits_list.append(mdl(x))
            labels_list.append(y)
    return torch.cat(logits_list), torch.cat(labels_list)


def compute_loss_for_indices(mdl, dataset, indices, dev, batch_size=512):
    """Compute per-example cross-entropy loss for given dataset indices."""
    all_losses = []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            bi = indices[s:s+batch_size]
            imgs, lbls = zip(*(dataset[i] for i in bi))
            x = torch.stack(imgs).to(dev)
            y = torch.tensor(lbls, dtype=torch.long, device=dev)
            losses = F.cross_entropy(mdl(x), y, reduction="none")
            all_losses.append(losses.cpu())
    return torch.cat(all_losses).numpy()


def evaluate(member_scores, nonmember_scores, label=""):
    """Run both point and conservative estimates, print results."""
    gap = statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)

    est_point = estimate_empirical_lower_bound(
        member_scores=member_scores, nonmember_scores=nonmember_scores,
        delta=prepare.DELTA, align_event_to_score_direction=True,
        require_member_favoring=True, report_confidence_supported_lower_bound=False,
    )
    est_cons = estimate_empirical_lower_bound(
        member_scores=member_scores, nonmember_scores=nonmember_scores,
        delta=prepare.DELTA, align_event_to_score_direction=True,
        require_member_favoring=True, report_confidence_supported_lower_bound=True,
    )

    tr_point = est_point.epsilon_lower_empirical / EPS_UPPER if EPS_UPPER > 0 else 0
    tr_cons = est_cons.epsilon_lower_empirical / EPS_UPPER if EPS_UPPER > 0 else 0

    print(f"  {label:40s} | gap={gap:+.4f} | point: eps={est_point.epsilon_lower_empirical:.4f} tr={tr_point:.4f} | "
          f"cons: eps={est_cons.epsilon_lower_empirical:.4f} tr={tr_cons:.4f} | "
          f"tpr={est_point.selected_tpr} fpr={est_point.selected_fpr} | "
          f"n={len(member_scores)}+{len(nonmember_scores)}")

    return {
        "label": label, "gap": gap,
        "eps_point": est_point.epsilon_lower_empirical,
        "eps_cons": est_cons.epsilon_lower_empirical,
        "tr_point": tr_point, "tr_cons": tr_cons,
        "tpr": est_point.selected_tpr, "fpr": est_point.selected_fpr,
        "n_member": len(member_scores), "n_nonmember": len(nonmember_scores),
    }


def train_shadow(train_dataset, eval_dataset, shadow_seed, epochs=1):
    """Train one shadow DP-SGD model on the given subset. Returns model in eval mode."""
    subset_bundle = DatasetBundle(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        input_dim=prepare.INPUT_DIM,
        num_classes=prepare.NUM_CLASSES,
        train_size=len(train_dataset),
        eval_size=len(eval_dataset),
    )
    config = prepare._build_train_config(epochs=epochs, seed=shadow_seed)
    set_global_seed(shadow_seed)
    logger = logging.getLogger(f"shadow_{shadow_seed}")
    outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=subset_bundle)
    sm = load_model_for_inference(config.model, outcome.checkpoint_path, device=device)
    return sm


print("Helpers loaded.")

## 2. Train K=64 shadow models

We train the maximum number of shadow models (K=64) upfront. For smaller K sweeps,
we simply use the first K models. Each shadow model trains on a random 50% subset
of the training data, with DP-SGD (same hyperparameters as the target model).

Expected time: ~7s per model on L4 GPU = ~7.5 minutes total.

In [ ]:
K_MAX = 64  # maximum number of shadow models
n_train = len(bundle.train_dataset)
half = n_train // 2

shadow_models = []
shadow_member_sets = []  # shadow_member_sets[k] = set of training indices included in shadow k

print(f"Training {K_MAX} shadow models (each on 50% of {n_train} training examples)...")
t0_all = time.time()

for k in range(K_MAX):
    t0 = time.time()
    shadow_seed = 5000 + k
    rng_k = random.Random(shadow_seed)

    # Random 50% subset of training indices
    all_indices = list(range(n_train))
    rng_k.shuffle(all_indices)
    in_indices = sorted(all_indices[:half])
    shadow_member_sets.append(set(in_indices))

    subset = Subset(bundle.train_dataset, in_indices)
    sm = train_shadow(subset, bundle.eval_dataset, shadow_seed)
    shadow_models.append(sm)

    elapsed = time.time() - t0
    if k < 3 or (k + 1) % 8 == 0:
        print(f"  Shadow {k:2d}/{K_MAX}: {elapsed:.1f}s")

total_time = time.time() - t0_all
print(f"\nDone. {K_MAX} shadow models trained in {total_time:.0f}s ({total_time/60:.1f} min)")

## 3. Precompute loss matrix

For efficiency, we precompute the loss of every shadow model on every target example
we will ever query. This avoids redundant forward passes across the K/budget/seed sweeps.

We build two matrices:
- `loss_matrix_train[k, i]` = loss of shadow model k on training example i
- `loss_matrix_eval[k, j]` = loss of shadow model k on eval example j

We also precompute the target model's losses on the same examples.

In [ ]:
# Collect all unique indices we will query across all seeds and budgets
SEEDS = [401, 402, 403, 404, 405]
BUDGETS = [128, 256, 512]
MAX_BUDGET = max(BUDGETS)

# Gather all member (training) and non-member (eval) indices across all seeds
all_train_indices = set()
all_eval_indices = set()
for seed in SEEDS:
    rng = random.Random(seed)
    mi = rng.sample(range(n_train), MAX_BUDGET // 2)
    ni = rng.sample(range(len(bundle.eval_dataset)), MAX_BUDGET // 2)
    all_train_indices.update(mi)
    all_eval_indices.update(ni)

all_train_indices = sorted(all_train_indices)
all_eval_indices = sorted(all_eval_indices)

# Index maps for fast lookup: original_index -> position in our arrays
train_idx_to_pos = {idx: pos for pos, idx in enumerate(all_train_indices)}
eval_idx_to_pos = {idx: pos for pos, idx in enumerate(all_eval_indices)}

print(f"Unique training indices to query: {len(all_train_indices)}")
print(f"Unique eval indices to query: {len(all_eval_indices)}")

# Precompute loss matrices: shape (K_MAX, num_unique_indices)
print(f"\nPrecomputing loss matrix for {K_MAX} shadow models...")
t0 = time.time()

loss_matrix_train = np.zeros((K_MAX, len(all_train_indices)), dtype=np.float32)
loss_matrix_eval = np.zeros((K_MAX, len(all_eval_indices)), dtype=np.float32)

for k, sm in enumerate(shadow_models):
    loss_matrix_train[k] = compute_loss_for_indices(sm, bundle.train_dataset, all_train_indices, device)
    loss_matrix_eval[k] = compute_loss_for_indices(sm, bundle.eval_dataset, all_eval_indices, device)
    if k < 2 or (k + 1) % 16 == 0:
        print(f"  Shadow {k:2d}/{K_MAX} done")

# Target model losses
target_loss_train = compute_loss_for_indices(model, bundle.train_dataset, all_train_indices, device)
target_loss_eval = compute_loss_for_indices(model, bundle.eval_dataset, all_eval_indices, device)

print(f"Loss matrices computed in {time.time()-t0:.1f}s")
print(f"  loss_matrix_train shape: {loss_matrix_train.shape}")
print(f"  loss_matrix_eval shape:  {loss_matrix_eval.shape}")

## 4. Gaussian LiRA scoring function

**The key innovation over v1.** For each target example with target model loss `l`:

1. Partition the K shadow models into IN-set (those that trained on this example)
   and OUT-set (those that did not).
2. Fit Gaussian to IN losses: N(mu_in, sigma_in^2)
3. Fit Gaussian to OUT losses: N(mu_out, sigma_out^2)
4. Score = log pdf_out(l) - log pdf_in(l)

Members should have higher scores: the target model loss is more likely under the
OUT distribution (because the target model learned this example, so its loss is
unusually low compared to models that didn't see it).

For non-members (eval set), all shadow models are OUT. We use a global OUT
distribution from all shadow losses on that example, and compare against the
target model loss using a one-sided approach (how unlikely is this loss under
the shadow distribution).

In [ ]:
SIGMA_FLOOR = 1e-6  # prevent division by zero in Gaussian pdf


def gaussian_log_pdf(x, mu, sigma):
    """Log probability density of x under N(mu, sigma^2)."""
    sigma = max(sigma, SIGMA_FLOOR)
    return -0.5 * np.log(2 * np.pi) - np.log(sigma) - 0.5 * ((x - mu) / sigma) ** 2


def compute_gaussian_lira_scores_member(train_indices, K, seed):
    """Compute Gaussian LiRA scores for member (training) examples.

    For each member example idx:
    - Partition first K shadow models into IN (trained on idx) and OUT (didn't)
    - Fit Gaussian to IN losses and OUT losses
    - Score = log_pdf_out(target_loss) - log_pdf_in(target_loss)
    """
    scores = []
    for idx in train_indices:
        pos = train_idx_to_pos[idx]
        target_loss = float(target_loss_train[pos])

        # Partition shadow models 0..K-1 into IN and OUT for this example
        in_losses = []
        out_losses = []
        for k in range(K):
            loss_k = float(loss_matrix_train[k, pos])
            if idx in shadow_member_sets[k]:
                in_losses.append(loss_k)
            else:
                out_losses.append(loss_k)

        # Need at least 2 samples for each group to fit a Gaussian
        if len(in_losses) >= 2 and len(out_losses) >= 2:
            mu_in, sigma_in = np.mean(in_losses), np.std(in_losses, ddof=1)
            mu_out, sigma_out = np.mean(out_losses), np.std(out_losses, ddof=1)
            score = gaussian_log_pdf(target_loss, mu_out, sigma_out) - \
                    gaussian_log_pdf(target_loss, mu_in, sigma_in)
        elif len(out_losses) >= 1 and len(in_losses) >= 1:
            # Fallback to raw mean difference
            score = np.mean(out_losses) - np.mean(in_losses)
        else:
            score = 0.0

        scores.append(score)
    return scores


def compute_gaussian_lira_scores_nonmember(eval_indices, K, seed):
    """Compute Gaussian LiRA scores for non-member (eval) examples.

    Non-members were not in any shadow model's training set, so all K shadow
    models are OUT. We use the shadow loss distribution as the OUT distribution
    and compute how likely the target loss is under it.

    To make scores comparable with members, we use:
    score = -|target_loss - mu_out| / sigma_out  (negative z-score)

    The idea: for non-members, target loss should be typical under the shadow
    distribution, giving z-scores near 0 (so scores near 0).
    For members (computed separately), target loss is unusually low under OUT,
    giving the Gaussian LiRA score a positive push.

    Alternative approach: use a global IN distribution estimated from the
    overall shadow loss statistics to compute a pseudo likelihood ratio.
    """
    scores = []
    for idx in eval_indices:
        pos = eval_idx_to_pos[idx]
        target_loss = float(target_loss_eval[pos])

        # All K shadow models are OUT for non-members
        out_losses = [float(loss_matrix_eval[k, pos]) for k in range(K)]

        if len(out_losses) >= 2:
            mu_out = np.mean(out_losses)
            sigma_out = np.std(out_losses, ddof=1)

            # Use the global IN shift: on average, IN losses are slightly lower
            # than OUT losses. We estimate this shift from the training examples.
            # For the score, we use a synthetic IN distribution shifted from OUT.
            #
            # Actually, the cleaner approach from Carlini et al.: for non-members,
            # the target model behaves like an OUT model. So we compute the same
            # likelihood ratio but the target loss should look like it came from
            # the OUT distribution, giving a score near 0.
            #
            # Score = log_pdf(target_loss | OUT) - log_pdf(target_loss | IN)
            # where IN is estimated as N(mu_out - global_shift, sigma_out)
            # But this adds a parameter. Simpler: use the same formula with
            # a "virtual IN" that shifts the mean down by global_in_out_shift.
            #
            # Simplest correct approach: just use log_pdf_out(target_loss).
            # Members get log_pdf_out - log_pdf_in (two-sided).
            # Non-members get just log_pdf_out (one-sided, no IN data).
            # This is what Carlini et al. recommend for the "offline" setting.
            score = gaussian_log_pdf(target_loss, mu_out, sigma_out)
        else:
            score = 0.0

        scores.append(score)
    return scores


def compute_raw_lira_scores_member(train_indices, K, seed):
    """Compute raw mean-difference LiRA scores (v1 baseline) for members."""
    scores = []
    for idx in train_indices:
        pos = train_idx_to_pos[idx]

        in_losses = []
        out_losses = []
        for k in range(K):
            loss_k = float(loss_matrix_train[k, pos])
            if idx in shadow_member_sets[k]:
                in_losses.append(loss_k)
            else:
                out_losses.append(loss_k)

        if in_losses and out_losses:
            score = np.mean(out_losses) - np.mean(in_losses)
        elif out_losses:
            score = np.mean(out_losses)
        else:
            score = -np.mean(in_losses) if in_losses else 0.0
        scores.append(score)
    return scores


def compute_raw_lira_scores_nonmember(eval_indices, K, seed):
    """Compute raw LiRA scores (v1 baseline) for non-members."""
    scores = []
    for idx in eval_indices:
        pos = eval_idx_to_pos[idx]
        target_loss = float(target_loss_eval[pos])

        shadow_losses = [float(loss_matrix_eval[k, pos]) for k in range(K)]
        shadow_mean = np.mean(shadow_losses) if shadow_losses else 0.0
        score = shadow_mean - target_loss
        scores.append(score)
    return scores


print("Gaussian LiRA scoring functions defined.")

## 5. Full sweep: Gaussian LiRA vs Raw LiRA across K and budget

For each combination of (K, budget, scoring method), we:
1. Sample member + non-member indices using the 5 seeds
2. Compute scores using both Gaussian and raw scoring
3. Run the evaluate() function to get tightness ratios

In [ ]:
K_VALUES = [8, 16, 32, 64]
results = []

print("=" * 110)
print("FULL SWEEP: Gaussian LiRA vs Raw LiRA")
print("=" * 110)

for K in K_VALUES:
    print(f"\n--- K = {K} shadow models ---")
    for budget in BUDGETS:
        # Accumulate scores across all seeds
        gauss_m, gauss_n = [], []
        raw_m, raw_n = [], []

        for seed in SEEDS:
            rng = random.Random(seed)
            mi = rng.sample(range(n_train), budget // 2)
            ni = rng.sample(range(len(bundle.eval_dataset)), budget // 2)

            # Gaussian LiRA
            gauss_m.extend(compute_gaussian_lira_scores_member(mi, K, seed))
            gauss_n.extend(compute_gaussian_lira_scores_nonmember(ni, K, seed))

            # Raw LiRA (v1 baseline)
            raw_m.extend(compute_raw_lira_scores_member(mi, K, seed))
            raw_n.extend(compute_raw_lira_scores_nonmember(ni, K, seed))

        # Evaluate both
        r_gauss = evaluate(gauss_m, gauss_n, label=f"Gauss K={K} b={budget}")
        r_gauss["method"] = "gaussian"
        r_gauss["K"] = K
        r_gauss["budget"] = budget
        results.append(r_gauss)

        r_raw = evaluate(raw_m, raw_n, label=f"Raw   K={K} b={budget}")
        r_raw["method"] = "raw"
        r_raw["K"] = K
        r_raw["budget"] = budget
        results.append(r_raw)

print("\n" + "=" * 110)
print("Sweep complete.")

## 6. Add standard baselines for comparison

Run neg_loss and correct_logit baselines (no shadow models needed) so we can
see the full picture in one table.

In [ ]:
print("\n=== Standard baselines (no shadow models) ===")

score_fns = {
    "neg_loss": lambda lg, lb: (-F.cross_entropy(lg, lb, reduction="none")).cpu().tolist(),
    "correct_logit": lambda lg, lb: lg.gather(1, lb.unsqueeze(1)).squeeze().cpu().tolist(),
}

for sname, sfn in score_fns.items():
    for budget in BUDGETS:
        all_m, all_n = [], []
        for seed in SEEDS:
            rng = random.Random(seed)
            mi = rng.sample(range(len(bundle.train_dataset)), budget // 2)
            ni = rng.sample(range(len(bundle.eval_dataset)), budget // 2)
            ml, mlb = collect_logits(model, bundle.train_dataset, mi, device)
            nl, nlb = collect_logits(model, bundle.eval_dataset, ni, device)
            all_m.extend(sfn(ml, mlb))
            all_n.extend(sfn(nl, nlb))
        r = evaluate(all_m, all_n, label=f"{sname} b={budget}")
        r["method"] = sname
        r["K"] = 0
        r["budget"] = budget
        results.append(r)

## 7. Diagnostics: Score distributions for best Gaussian LiRA config

Visualize the member vs non-member score distributions to understand
how well the Gaussian LiRA separates the two populations.

In [ ]:
import matplotlib.pyplot as plt

# Recompute scores for the largest config: K=64, budget=512
diag_K = 64
diag_budget = 512
diag_gauss_m, diag_gauss_n = [], []
diag_raw_m, diag_raw_n = [], []

for seed in SEEDS:
    rng = random.Random(seed)
    mi = rng.sample(range(n_train), diag_budget // 2)
    ni = rng.sample(range(len(bundle.eval_dataset)), diag_budget // 2)
    diag_gauss_m.extend(compute_gaussian_lira_scores_member(mi, diag_K, seed))
    diag_gauss_n.extend(compute_gaussian_lira_scores_nonmember(ni, diag_K, seed))
    diag_raw_m.extend(compute_raw_lira_scores_member(mi, diag_K, seed))
    diag_raw_n.extend(compute_raw_lira_scores_nonmember(ni, diag_K, seed))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gaussian LiRA
ax = axes[0]
ax.hist(diag_gauss_m, bins=50, alpha=0.6, label=f"Members (n={len(diag_gauss_m)})", color="tab:blue", density=True)
ax.hist(diag_gauss_n, bins=50, alpha=0.6, label=f"Non-members (n={len(diag_gauss_n)})", color="tab:red", density=True)
ax.set_xlabel("Gaussian LiRA score")
ax.set_ylabel("Density")
ax.set_title(f"Gaussian LiRA (K={diag_K}, budget={diag_budget})")
ax.legend()
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

# Raw LiRA
ax = axes[1]
ax.hist(diag_raw_m, bins=50, alpha=0.6, label=f"Members (n={len(diag_raw_m)})", color="tab:blue", density=True)
ax.hist(diag_raw_n, bins=50, alpha=0.6, label=f"Non-members (n={len(diag_raw_n)})", color="tab:red", density=True)
ax.set_xlabel("Raw LiRA score (mean OUT - mean IN)")
ax.set_ylabel("Density")
ax.set_title(f"Raw LiRA (K={diag_K}, budget={diag_budget})")
ax.legend()
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(str(WORK / "autoresearch" / "results" / "lira_v2_score_distributions.png"), dpi=150)
plt.show()

print(f"\nGaussian LiRA K={diag_K}:")
print(f"  Member scores:     mean={np.mean(diag_gauss_m):.4f}, std={np.std(diag_gauss_m):.4f}")
print(f"  Non-member scores: mean={np.mean(diag_gauss_n):.4f}, std={np.std(diag_gauss_n):.4f}")
print(f"  Gap: {np.mean(diag_gauss_m) - np.mean(diag_gauss_n):.4f}")

print(f"\nRaw LiRA K={diag_K}:")
print(f"  Member scores:     mean={np.mean(diag_raw_m):.4f}, std={np.std(diag_raw_m):.4f}")
print(f"  Non-member scores: mean={np.mean(diag_raw_n):.4f}, std={np.std(diag_raw_n):.4f}")
print(f"  Gap: {np.mean(diag_raw_m) - np.mean(diag_raw_n):.4f}")

## 8. Results summary and CSV export

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

# Sort by conservative tightness (the metric that matters), then point
df = df.sort_values(["tr_cons", "tr_point"], ascending=False)

print("=" * 120)
print(f"ALL RESULTS -- epsilon_upper = {EPS_UPPER:.4f}")
print("=" * 120)
print(df[["label", "method", "K", "budget", "gap", "eps_point", "tr_point",
          "eps_cons", "tr_cons", "tpr", "fpr", "n_member"]].to_string(index=False))

# Highlight best results
print("\n" + "-" * 80)
print("BEST CONSERVATIVE (Wilson CI) tightness:")
cons_best = df[df["tr_cons"] > 0].head(5)
if len(cons_best) > 0:
    for _, row in cons_best.iterrows():
        print(f"  {row['label']:40s}  tr_cons={row['tr_cons']:.4f}  eps_cons={row['eps_cons']:.4f}")
else:
    print("  None -- all conservative estimates are 0")

print("\nBEST POINT-ESTIMATE tightness:")
for _, row in df.head(5).iterrows():
    print(f"  {row['label']:40s}  tr_point={row['tr_point']:.4f}  eps_point={row['eps_point']:.4f}")

# Gaussian vs Raw comparison at each K
print("\n" + "-" * 80)
print("GAUSSIAN vs RAW LiRA comparison (budget=512, largest sample):")
for K in K_VALUES:
    gauss_row = df[(df["method"] == "gaussian") & (df["K"] == K) & (df["budget"] == 512)]
    raw_row = df[(df["method"] == "raw") & (df["K"] == K) & (df["budget"] == 512)]
    if len(gauss_row) > 0 and len(raw_row) > 0:
        g = gauss_row.iloc[0]
        r = raw_row.iloc[0]
        print(f"  K={K:2d}: Gaussian tr_point={g['tr_point']:.4f} tr_cons={g['tr_cons']:.4f}  |  "
              f"Raw tr_point={r['tr_point']:.4f} tr_cons={r['tr_cons']:.4f}")

# Save CSV
out_dir = WORK / "autoresearch" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "lira_v2_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nResults saved to {csv_path}")

## 9. Download results

In [ ]:
from google.colab import files

files.download(str(csv_path))
files.download(str(WORK / "autoresearch" / "results" / "lira_v2_score_distributions.png"))
print("Downloads triggered.")